# AI Code Reviewer — end-to-end demo

This notebook runs the exact same pipeline as the CLI (`python -m ai_code_reviewer.cli sample_code/example.py --review`), but step by step so you can see each stage's output inline: static analyzers -> normalized findings -> LLM call -> structured review.

Run this notebook from the `ai-code-reviewer` project root using the project's `venv` kernel (select it in VS Code's kernel picker, top-right of the notebook).

## 1. The file being reviewed

In [5]:
TARGET = "sample_code/iris_logistic_regression.py"

with open(TARGET, encoding="utf-8") as f:
    print(f.read())

# Step 1: Import libraries
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

# Step 2: Load dataset
data = load_iris()
X = data.data
y = data.target

# Step 3: Split dataset
X_train, y_train, X_test, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 4: Create model
model = LogisticRegression(max_iter=200)

# Step 5: Train model
model.fit(X_train, y_train)

# Step 6: Predict
y_pred = model.predict(X_test)

# Step 7: Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)



## 2. Static analysis (pylint + flake8 + bandit), normalized

In [6]:
from ai_code_reviewer.analyzers import analyze

findings = analyze(TARGET)
findings

[{'tool': 'flake8',
  'file': 'sample_code/iris_logistic_regression.py',
  'line': 2,
  'column': 1,
  'code': 'F401',
  'message': "'numpy as np' imported but unused",
  'severity': 'critical'},
 {'tool': 'pylint',
  'file': 'sample_code\\iris_logistic_regression.py',
  'line': 1,
  'column': 0,
  'code': 'C0114',
  'message': 'Missing module docstring',
  'severity': 'suggestion'},
 {'tool': 'pylint',
  'file': 'sample_code\\iris_logistic_regression.py',
  'line': 2,
  'column': 0,
  'code': 'W0611',
  'message': 'Unused numpy imported as np',
  'severity': 'warning'},
 {'tool': 'pylint',
  'file': 'sample_code\\iris_logistic_regression.py',
  'line': 14,
  'column': 0,
  'code': 'C0103',
  'message': 'Constant name "X_train" doesn\'t conform to UPPER_CASE naming style',
  'severity': 'suggestion'},
 {'tool': 'pylint',
  'file': 'sample_code\\iris_logistic_regression.py',
  'line': 14,
  'column': 18,
  'code': 'C0103',
  'message': 'Constant name "X_test" doesn\'t conform to UPPER_C

## 3. LLM review (OpenAI gpt-4o, structured JSON output)

Requires `OPENAI_API_KEY` in `.env` at the project root (already configured).

In [7]:
from ai_code_reviewer.llm_reviewer import review_file

review = review_file(TARGET)
review

{'summary': 'The file implements logistic regression on the Iris dataset with acceptable clarity but several issues need addressing, including critical import issues and naming inconsistencies. Overall, the code is functional but needs refinement for best practices.',
 'comments': [{'line': 2,
   'severity': 'critical',
   'title': "Unused import 'numpy as np'",
   'comment': "The import statement 'import numpy as np' is not used anywhere in the code. Unused imports should be removed to avoid clutter and reduce unnecessary dependencies, improving performance slightly by not loading unused modules."},
  {'line': 1,
   'severity': 'suggestion',
   'title': 'Missing module docstring',
   'comment': "A docstring at the beginning of the file would enhance clarity by providing a brief overview of the script's purpose, inputs, and outputs. This helps with maintenance and understanding by other developers."},
  {'line': 14,
   'severity': 'critical',
   'title': 'Incorrect arguments to train_t

## 4. Pretty-printed final review (same formatting the CLI uses)

In [8]:
severity_order = {"critical": 0, "warning": 1, "suggestion": 2}

print(f"Summary: {review['summary']}\n")
for c in sorted(review["comments"], key=lambda c: severity_order.get(c["severity"], 3)):
    print(f"[{c['severity'].upper()}] line {c['line']}: {c['title']}")
    print(f"  {c['comment']}\n")

Summary: The file implements logistic regression on the Iris dataset with acceptable clarity but several issues need addressing, including critical import issues and naming inconsistencies. Overall, the code is functional but needs refinement for best practices.

[CRITICAL] line 2: Unused import 'numpy as np'
  The import statement 'import numpy as np' is not used anywhere in the code. Unused imports should be removed to avoid clutter and reduce unnecessary dependencies, improving performance slightly by not loading unused modules.

[CRITICAL] line 14: Incorrect arguments to train_test_split
  The call to 'train_test_split' has inputs in the wrong order; 'train_test_split' should be called with 'X_train', 'X_test', 'y_train', 'y_test = train_test_split(X, y, ...)'. The current arguments will cause a runtime error as 'X_train' and 'y_train' are meant to be used for the training split, but 'X_test' and 'y_test' are mishandling the split operation (variables misnamed).

[SUGGESTION] line 